## Data Cleaning Script

### Packages

In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd
import requests
from io import StringIO
from shapely import wkt, distance, LineString

### Reading the Data

In [2]:
url = "https://raw.githubusercontent.com/iansargent/nyc-subway-ridership-ml/refs/heads/main/Data/Raw/origin_destination_flows.csv"
od_flows_raw = pd.read_csv(StringIO(requests.get(url, verify=False).text))
od_flows = od_flows_raw.copy()

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'raw.githubusercontent.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


### Cleaning

#### Column Names

In [3]:
# Standardize column names
od_flows.columns = od_flows_raw.columns.str.strip().str.lower().str.replace(" ", "_")

#### Data Types

In [4]:
# Turn the ridership column to numeric (remove commas)
od_flows['sum_estimated_average_ridership'] = (
    pd.to_numeric(od_flows_raw['sum_estimated_average_ridership'].str.replace(',', ''))
)

#### Turn the Point Data into a Geometry Type

In [5]:
od_flows["origin_point"] = od_flows["origin_point"].apply(wkt.loads)
od_flows["destination_point"] = od_flows["destination_point"].apply(wkt.loads)

#### Turn into a GeoDataFrame for Mapping

In [7]:
od_flows["geometry"] = od_flows.apply(
    lambda row: LineString([
        row["origin_point"],
        row["destination_point"]
    ]),
    axis=1
)

od_flows_geo = gpd.GeoDataFrame(
    od_flows,
    geometry="geometry",
    crs="EPSG:4326"
)

#### Calculate Distance Between Two Stations (Meters + Kilometers)

In [8]:
od_projected = od_flows_geo.to_crs("EPSG:32618")

od_flows_geo["distance_meters"] = od_projected.length
od_flows_geo["distance_km"] = od_flows_geo["distance_meters"] / 1000

#### Log of Ridership (skewed)

In [9]:
od_flows_geo["log_ridership"] = np.log1p(od_flows_geo["sum_estimated_average_ridership"])

#### Riders per Kilometer Variable

In [11]:
od_flows_geo["riders_per_km"] = (
    od_flows_geo["sum_estimated_average_ridership"] /
    od_flows_geo["distance_km"]
)

In [15]:
od_flows_geo["geometry"] = od_flows_geo.apply(
    lambda row: LineString([row["origin_point"], row["destination_point"]]), axis=1
)

od_flows_geo["origin_point_wkt"] = od_flows_geo["origin_point"].apply(lambda g: g.wkt)
od_flows_geo["destination_point_wkt"] = od_flows_geo["destination_point"].apply(lambda g: g.wkt)

od_flows_geo.drop(columns=["origin_point", "destination_point"]) \
    .set_geometry("geometry") \
    .to_parquet("origin_destination_flows_CLEAN.parquet")